In [ ]:
import pandas as pd

df = pd.read_csv("../out.csv")

df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["Bioma", "DiaSemChuva", "Precipitacao", "RiscoFogo"]]
y = df["Fogo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(8),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}\n")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessClassifier

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(
    length_scale=[1.0] * X_train_scaled.shape[1], 
    length_scale_bounds=(1e-2, 1e2)
)

gpc = GaussianProcessClassifier(
    kernel=kernel,
    random_state=42,
    n_jobs=-1
)

gpc.fit(X_train_scaled, y_train)

y_pred = gpc.predict(X_test_scaled)
y_prob = gpc.predict_proba(X_test_scaled)

print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}\n")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

lr = LogisticRegression(
    random_state=42,
    n_jobs=-1,
    max_iter=1000
)

lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)

print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}\n")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
teste = pd.DataFrame([
    {
        "Bioma": 1.7,
        "DiaSemChuva": 23,
        "Precipitacao": 0,
        "RiscoFogo": 1
    }
])

teste_s = scaler.transform(teste)

print(f'Probabilidade de incêndio (MLP): {mlp.predict_proba(teste_s)[0][1]:.2f}')
print(f'Probabilidade de incêndio (GPC): {gpc.predict_proba(teste_s)[0][1]:.2f}')
print(f'Probabilidade de incêndio (LR): {lr.predict_proba(teste_s)[0][1]:.2f}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Garante a conversão para array NumPy se forem DataFrames do Pandas
X_tr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else X_train
X_te = X_test.to_numpy() if hasattr(X_test, "to_numpy") else X_test
y_tr = y_train.to_numpy() if hasattr(y_train, "to_numpy") else y_train
y_te = y_test.to_numpy() if hasattr(y_test, "to_numpy") else y_test

# Setup de colunas e features
features = ["bioma", "diassemchuva", "precipitacao", "riscofogo"]
rf_idx = 3  # Índice do riscofogo
other_indices = [0, 1, 2]  # bioma, diassemchuva, precipitacao

mean_unscaled = X_tr.mean(axis=0)

# Alterado para 3 linhas (3x3 subplots) e ajustado o tamanho da figura
fig, axes = plt.subplots(3, 3, figsize=(16, 14))

models_info = [
    ("MLP", mlp, axes[0]), 
    ("GPC", gpc, axes[1]),
    ("Regressão Logística", lr, axes[2])  # Adicionado o modelo Random Forest
]

for model_name, model, row_axes in models_info:
    for i, other_idx in enumerate(other_indices):
        ax = row_axes[i]

        # Limites na escala original das features
        x_min, x_max = X_tr[:, rf_idx].min() - 0.5, X_tr[:, rf_idx].max() + 0.5
        y_min, y_max = (
            X_tr[:, other_idx].min() - 0.5,
            X_tr[:, other_idx].max() + 0.5,
        )

        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100)
        )

        # Matriz para predizer probabilidade
        grid_unscaled = np.tile(mean_unscaled, (xx.ravel().shape[0], 1))
        grid_unscaled[:, rf_idx] = xx.ravel()
        grid_unscaled[:, other_idx] = yy.ravel()
        grid_scaled = scaler.transform(grid_unscaled)

        Z = model.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

        # Plot
        ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap=plt.cm.RdBu, vmin=0, vmax=1, alpha=0.75)
        ax.scatter(
            X_te[:, rf_idx],
            X_te[:, other_idx],
            c=y_te,
            cmap=plt.cm.RdBu,   # mesmo cmap do fundo
            vmin=0, vmax=1,
            edgecolors="k",
            alpha=0.7,
        )

        ax.set_xlabel("riscofogo")
        ax.set_ylabel(features[other_idx])
        ax.set_title(f"{model_name}: riscofogo vs {features[other_idx]}")

        acc = model.score(X_test_scaled, y_te)
        ax.text(
            0.95,
            0.05,
            f"Test Acc: {acc:.2f}",
            transform=ax.transAxes,
            size=11,
            horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# df is your pandas DataFrame
corr = df.select_dtypes(include="number").corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()
